Import das bibliotecas necessárias

In [1]:
from docx import Document

import pandas as pd

import os

import smtplib

import datetime

Criaçao das pastas necessárias para o projeto

In [2]:
os.makedirs("modelo", exist_ok=True )

os.makedirs("Planilhas", exist_ok=True)

os.makedirs("Processados", exist_ok=True)

os.makedirs("Processados", exist_ok=True)

os.makedirs("Erros", exist_ok=True)

print("Pastas criadas no sistema:")

lista_pasta_criadas = os.listdir()

for pasta in lista_pasta_criadas:

    if '.' not in pasta:

        print(pasta)



Pastas criadas no sistema:
Erros
LICENSE
modelo
Planilhas
Processados


Criando o caminho de acesso para a nossa pasta de planilhas

In [2]:
pasta_planilhas = os.listdir('Planilhas')

pasta_planilhas

['exercicio d de bd.brM',
 'Lista 1 de candidatos.xlsx',
 'Lista 2 de candidatos.xlsx']

caminho do modelo gerado pelo RH

In [3]:
modelo_rh = os.listdir('modelo') 

modelo_rh

['modelo contratação.docx']

Acessando a pasta de planilhas

In [4]:

for arquivo in pasta_planilhas:

  try:
    
      if arquivo.endswith('.xlsx'):

         nomes_colunas = ["nome", "cpf", "endereco", "cargo", "salario", "cidade-uf", "data de inicio", "email"]

         df_planilhas = pd.read_excel(f"Planilhas/{arquivo}", header=0, names=nomes_colunas)

         cpf = df_planilhas['cpf'].fillna("")

         cpf_limpo = cpf.astype(str, errors='ignore')

         cpf_invalidos = []

         for cpf_candidato in cpf_limpo:

            total_cpf = (cpf_limpo == cpf_candidato).sum()
            
            if len(cpf_candidato) != 14:

                print(f"O cpf {cpf_candidato} da planilha {arquivo} não possui 11 digitos")

                with open(f"Erros/erros_{arquivo}.txt", "a", encoding='utf-8') as erro_log:

                  erro_log.write(f"O cpf {cpf_candidato} da planilha {arquivo} não possui 11 digitos\n")

                  cpf_invalidos.append(cpf_candidato)

            elif total_cpf > 1:

              print(f"O cpf {cpf_candidato} da planilha {arquivo} está presente em mais de um registro")

              with open(f"Erros/erros_{arquivo}.txt", "a", encoding='utf-8') as erro_log:

                erro_log.write(f"O {cpf_candidato} da planilha {arquivo} está presente em mais de um registro\n")

                cpf_invalidos.append(cpf_candidato)

         df_cpfs_validos = df_planilhas[~cpf_limpo.isin(cpf_invalidos)]  
         
         for index, linha in df_cpfs_validos.iterrows():

            documento = Document(f"modelo/{modelo_rh[0]}")

            nome = str(linha['nome'])

            cpf = str(linha['cpf'])

            endereco = str(linha['endereco'])

            cargo = str(linha['cargo'])

            salario = str(linha['salario'])

            cidade_uf = str(linha['cidade-uf'])

            data_emissao = datetime.datetime.now().strftime("%d/%m/%Y")

            data_inicio = str(linha['data de inicio'])

            email = (linha['email'])

            itens_contratos = {'[NOME_FUNCIONARIO]': nome, 
                              '[CPF_FUNCIONARIO]': cpf, 
                              '[ENDERECO_FUNCIONARIO]': endereco,
                              '[CARGO_FUNCIONARIO]':cargo, 
                              '[SALARIO_FUNCIONARIO]':salario, 
                              '[DATA_INICIO]':data_inicio,
                              '[CIDADE-UF]': cidade_uf,
                              '[DATA_EMISSÃO]': data_emissao}

            if nome == 'nome' and cpf == 'cpf' and endereco == 'endereco' and cargo == 'cargo' and salario == 'salario' and data_inicio == 'data de inicio' and cidade_uf == 'cidade-uf':

                continue                  

            for paragrafo in documento.paragraphs:

               for chave, valor in itens_contratos.items():

                  if chave in paragrafo.text:

                    paragrafo.text = paragrafo.text.replace(chave, valor)
            
            documento.save(f"Processados/contrato_{nome}.docx")
          
      else:

        print(f"O arquivo {arquivo} não é um excel")

  except FileNotFoundError as erro:

    print(f"O arquivo {arquivo} não foi encontrado: {erro}")

O arquivo exercicio d de bd.brM não é um excel
O cpf cpf da planilha Lista 1 de candidatos.xlsx não possui 11 digitos
O cpf 123.456.789-10 da planilha Lista 1 de candidatos.xlsx está presente em mais de um registro
O cpf 123.456.789-10 da planilha Lista 1 de candidatos.xlsx está presente em mais de um registro
O cpf 123.567.89 da planilha Lista 1 de candidatos.xlsx não possui 11 digitos
O cpf  da planilha Lista 2 de candidatos.xlsx não possui 11 digitos
O cpf cpf da planilha Lista 2 de candidatos.xlsx não possui 11 digitos
O cpf 234.5678.90 da planilha Lista 2 de candidatos.xlsx não possui 11 digitos
O cpf 234.5678.90 da planilha Lista 2 de candidatos.xlsx não possui 11 digitos
